# Bone-CVQA trên Kaggle

Notebook mẫu để clone project, gắn dataset, tải model, train/evaluate/inference và visualize kết quả.

**Khuyến nghị:** đưa dataset vào Kaggle Dataset rồi chọn **Add Input**. Dataset sẽ xuất hiện dưới `/kaggle/input/...`, ổn định hơn việc tải lại từ internet. Hugging Face phù hợp để tải model; nếu model lớn hoặc private, nên tải trước vào Kaggle Dataset/Model rồi dùng `local_dir`.

Nếu cần model Hugging Face gated/private, tạo Kaggle Secret tên `HF_TOKEN` tại **Add-ons -> Secrets**.

## 1. Clone project và cài dependency

Bật **Internet** trong Kaggle Settings nếu muốn `git clone`, cài package hoặc tải model.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HuyenHuyenUwU/cvqa-draft.git"
REPO_DIR = Path("/kaggle/working/cvqa-draft")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Project:", Path.cwd())

## 2. Chọn dataset và vị trí output

Sau khi chọn Kaggle Dataset ở **Add Input**, sửa `DATA_ROOT` theo thư mục trong `/kaggle/input`.

FracAtlas mặc định cần `images/Fractured`, `images/Non_fractured` và tùy chọn `Annotations/COCO JSON/COCO_fracture_masks.json`. BTXRD cần `images`, `dataset.csv` và cấu hình cột tương ứng trong `configs/data.yaml`.

In [ ]:
DATASET_NAME = "fracatlas"  # "fracatlas" hoặc "btxrd"
DATA_ROOT = Path("/kaggle/input/fracatlas")  # SỬA theo tên Kaggle Dataset
WORK_DIR = Path("/kaggle/working/cvqa_run")
OUTPUT_DIR = WORK_DIR / "outputs"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
LOG_DIR = WORK_DIR / "logs"
assert DATA_ROOT.exists(), f"Không tìm thấy DATA_ROOT: {DATA_ROOT}"
print("Dataset files:", sum(1 for p in DATA_ROOT.rglob("*") if p.is_file()))

In [ ]:
# Kiểm tra schema trước khi sửa YAML hoặc train.
inspection_path = WORK_DIR / "dataset_inspection.json"
inspection_path.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "python", "scripts/inspect_dataset.py",
    "--root", str(DATA_ROOT), "--out", str(inspection_path)
], check=True)
print("Full report:", inspection_path)

## 3. Model và API key

Dùng `tiny_random` để test pipeline không cần tải model lớn. Với model thật, đặt `allow_download: true` cho model public hoặc đặt `local_dir` tới model trong Kaggle Dataset và `allow_download: false`.

Cell dưới đọc Hugging Face token từ Kaggle Secret, không in token. `provider: api` của project hiện là placeholder, chưa gọi API trả phí.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN is configured (value hidden)")
    else:
        print("HF_TOKEN is empty; public/local models can still work")
except Exception as exc:
    print("HF_TOKEN not configured:", type(exc).__name__)

## 4. Train với tham số có thể chỉnh

Bắt đầu với `MAX_TRAIN_BATCHES=20` để kiểm tra đường ống, sau đó đặt `None` để train toàn bộ. Checkpoint được ghi vào `/kaggle/working/cvqa_run/checkpoints/best.pt`.

In [ ]:
EPOCHS = 2
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
DEVICE = "cuda"  # đổi thành "cpu" nếu GPU không khả dụng
MAX_TRAIN_BATCHES = 20  # đổi thành None để train toàn bộ
MAX_VAL_BATCHES = 5
NUM_WORKERS = 2
data_root_key = f"data.datasets.{DATASET_NAME}.root"
train_args = [
    "python", "scripts/train.py", "--configs", "base,data,model,training", "--set",
    "data.name=" + DATASET_NAME, data_root_key + "=" + str(DATA_ROOT),
    "device=" + DEVICE, "paths.output_dir=" + str(OUTPUT_DIR),
    "paths.checkpoint_dir=" + str(CHECKPOINT_DIR), "paths.log_dir=" + str(LOG_DIR),
    "train.epochs=" + str(EPOCHS), "train.batch_size=" + str(BATCH_SIZE),
    "train.lr=" + str(LEARNING_RATE), "train.num_workers=" + str(NUM_WORKERS),
    "train.amp=" + str(DEVICE == "cuda").lower(),
]
if MAX_TRAIN_BATCHES is not None:
    train_args.append("train.max_train_batches=" + str(MAX_TRAIN_BATCHES))
if MAX_VAL_BATCHES is not None:
    train_args.append("train.max_val_batches=" + str(MAX_VAL_BATCHES))
print(" ".join(train_args))
subprocess.run(train_args, check=True)

## 5. Evaluate và xem metric

Evaluation tạo `eval_test.json` và `predictions_test.csv` trong `OUTPUT_DIR`. Answer có thể được sinh bằng weak template, không nhất thiết là ground truth chuyên gia.

In [ ]:
import json
import pandas as pd
from IPython.display import display

CHECKPOINT = CHECKPOINT_DIR / "best.pt"
assert CHECKPOINT.exists(), f"Chưa có checkpoint: {CHECKPOINT}"
evaluate_args = [
    "python", "scripts/evaluate.py", "--configs", "base,data,model,training,inference",
    "--checkpoint", str(CHECKPOINT), "--split", "test", "--set",
    "data.name=" + DATASET_NAME, data_root_key + "=" + str(DATA_ROOT),
    "device=" + DEVICE, "paths.output_dir=" + str(OUTPUT_DIR),
    "inference.generation.max_new_tokens=48", "inference.generation.do_sample=false",
]
subprocess.run(evaluate_args, check=True)
metrics_path = OUTPUT_DIR / "eval_test.json"
predictions_path = OUTPUT_DIR / "predictions_test.csv"
if metrics_path.exists():
    display(pd.json_normalize(json.loads(metrics_path.read_text(encoding="utf-8"))))
if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    display(predictions.head(10))

## 6. Inference một ảnh và visualize

Cell tự tìm ảnh đầu tiên; có thể thay `IMAGE_PATH` bằng ảnh khác. JSON gồm answer, warning và các trạng thái verification.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

image_candidates = [p for ext in ("*.jpg", "*.jpeg", "*.png", "*.dcm")
                    for p in glob.glob(str(DATA_ROOT / "**" / ext), recursive=True)]
assert image_candidates, f"Không tìm thấy ảnh trong {DATA_ROOT}"
IMAGE_PATH = Path(image_candidates[0])
QUESTION = "Is there a fracture in this X-ray image?"
INFERENCE_OUTPUT = OUTPUT_DIR / "inference_result.json"
subprocess.run([
    "python", "scripts/infer.py", "--configs", "base,model,inference",
    "--checkpoint", str(CHECKPOINT), "--image", str(IMAGE_PATH),
    "--question", QUESTION, "--out", str(INFERENCE_OUTPUT),
    "--set", "device=" + DEVICE,
], check=True)
result = json.loads(INFERENCE_OUTPUT.read_text(encoding="utf-8"))
display(result)
if IMAGE_PATH.suffix.lower() != ".dcm":
    plt.figure(figsize=(7, 7))
    plt.imshow(Image.open(IMAGE_PATH), cmap="gray")
    plt.title(f"{IMAGE_PATH.name}\n{QUESTION}")
    plt.axis("off")
    plt.show()

## 7. Visualize prediction và lưu artifact

Đồ thị này visualize answer output, không phải heatmap lesion; localization/evidence graph trong baseline vẫn là placeholder.

In [ ]:
import matplotlib.pyplot as plt
import shutil
from IPython.display import FileLink, display

if predictions_path.exists():
    counts = predictions["prediction"].fillna("<empty>").value_counts().head(15)
    plt.figure(figsize=(10, 5))
    counts.sort_values().plot.barh()
    plt.title("Top predicted answers")
    plt.xlabel("count")
    plt.tight_layout()
    plt.show()
archive_path = Path("/kaggle/working/cvqa_run_artifacts.zip")
shutil.make_archive(str(archive_path.with_suffix("")), "zip", WORK_DIR)
print("Artifact archive:", archive_path)
display(FileLink(str(archive_path)))

## Gợi ý khi chạy lại

- Lần đầu dùng `MAX_TRAIN_BATCHES=5` và `EPOCHS=1` để kiểm tra dataset/model.
- Khi pipeline chạy đúng, đặt `MAX_TRAIN_BATCHES=None`, tăng `EPOCHS`, rồi **Save Version**.
- Nếu session bị reset, checkpoint trong `/kaggle/working` mất; tải file zip hoặc lưu output notebook.
- Không đưa `HF_TOKEN`, API key hoặc dữ liệu riêng tư vào git, cell output hay notebook đã chia sẻ.